In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import math
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from os.path import join as pjoin
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr, spearmanr, zscore, kendalltau
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/overrep_cell_corrs'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
pos_distances = np.arange(0, (reward_bin_size*4) + reward_bin_size, reward_bin_size) ## distance from reward locations
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 0.2 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians
chunks = 4 ## number of time bins for cell-cell correlations

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse. Get the reward over-representation cells and calculate their time-binned correlations across the session.

In [ ]:
## Set mouse information
## mc54 and mc51 are good example mice for Two-context and Multi-context on day 16
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

minian_path = pjoin(dpath, f'{experiment}/minian_results/{mouse}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata.assign_coords(cumulative_rewards=('frame', np.cumsum(sdata['water'].values)))
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Find the peak of each place field
pf_peaks = np.max(tuning_curves, axis=1)
## Find the spatial bins where each peak occurred
field_dist = np.zeros(pf_peaks.shape[0])
for idx, peak in enumerate(pf_peaks):
    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

## Find distance from reward locations
rw_one_dist = field_dist - first_rw_pos
rw_two_dist = field_dist - second_rw_pos
bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
sub_uids = neural_data['unit_id'][active_cells]
rw1_uids = neural_data['unit_id'][rw1_bool]
rw2_uids = neural_data['unit_id'][rw2_bool]

## Select activity, then separate into time_bin_size time bins
rw1_act = neural_data.sel(unit_id=rw1_uids) ## can swap out neural_data or sdata here to change for whole session or only running + correct direction
rw2_act = neural_data.sel(unit_id=rw2_uids)
rw1_binned = ctn.bin_activity(rw1_act.values, bin_size_seconds=time_bin_size, func=np.mean)
rw2_binned = ctn.bin_activity(rw2_act.values, bin_size_seconds=time_bin_size, func=np.mean)

## Get cell-cell correlations across chunks of time
rw1_corrs = ctn.time_binned_cell_cell_correlations(rw1_binned, chunks=chunks)
rw2_corrs = ctn.time_binned_cell_cell_correlations(rw2_binned, chunks=chunks)

## Bin data by amount of rewards earned in that time
bins = np.arange(0, neural_data.shape[1], int(time_bin_size * 30)) ## 30 frames per second
binned = np.split(neural_data['cumulative_rewards'].values, bins)
binned_cumulative_rewards = np.array([np.round(np.mean(bin)) for bin in binned if bin.size > 0])
total_rewards = np.sum(sdata['water'].values)

rw1_reward_corrs = ctn.reward_binned_cell_cell_correlations(rw1_binned, total_rewards=total_rewards, 
                                                            cumulative_rewards=binned_cumulative_rewards, rw_chunks=chunks)
rw2_reward_corrs = ctn.reward_binned_cell_cell_correlations(rw2_binned, total_rewards=total_rewards, 
                                                            cumulative_rewards=binned_cumulative_rewards, rw_chunks=chunks)

In [ ]:
## Plot the average correlation value across time bin for the population of neurons around reward 1 and reward 2
fig = pf.custom_graph_template(x_title='Time Bin', y_title='Correlation', titles=[f'{mouse}'])
xaxis = np.arange(chunks)
fig.add_trace(go.Scattergl(x=xaxis, y=np.nanmean(np.nanmean(rw1_corrs, axis=0), axis=0), mode='lines+markers', name=f'Reward 1 ({rw1_uids.shape[0]})',
                           marker=dict(line=dict(width=1.5, color='black')), marker_size=9, line_width=2.5, line_color='darkgrey'))
fig.add_trace(go.Scattergl(x=xaxis, y=np.nanmean(np.nanmean(rw2_corrs, axis=0), axis=0), mode='lines+markers', name=f'Reward 2 ({rw2_uids.shape[0]})',
                           marker=dict(line=dict(width=1.5, color='black')), marker_size=9, line_width=2.5, line_color=ce_colors_dict[sdata.attrs['group']]))
fig.update_yaxes(range=[-0.1, 0.8])
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_{cell_type}_rw1_rw2_pop_corr.png'), width=500, height=500)

In [ ]:
## Plot the average correlation value across reward chunks for the population of neurons around reward 1 and reward 2
fig = pf.custom_graph_template(x_title='Reward Bin', y_title='Correlation', titles=[f'{mouse}'])
xaxis = np.arange(chunks)
fig.add_trace(go.Scattergl(x=xaxis, y=np.nanmean(np.nanmean(rw1_reward_corrs, axis=0), axis=0), mode='lines+markers', name=f'Reward 1 ({rw1_uids.shape[0]})',
                           marker=dict(line=dict(width=1.5, color='black')), marker_size=9, line_width=2.5, line_color='darkgrey'))
fig.add_trace(go.Scattergl(x=xaxis, y=np.nanmean(np.nanmean(rw2_reward_corrs, axis=0), axis=0), mode='lines+markers', name=f'Reward 2 ({rw2_uids.shape[0]})',
                           marker=dict(line=dict(width=1.5, color='black')), marker_size=9, line_width=2.5, line_color=ce_colors_dict[sdata.attrs['group']]))
fig.update_yaxes(range=[-0.1, 0.8])
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_{cell_type}_rw1_rw2_pop_reward_corr.png'), width=500, height=500)